# Exploring the Structure of Romanian Deputies' Profiles

## Objective

Although the data on the Romanian Chamber of Deputies’ profile pages appears consistent at first glance, there is noticeable variation in:
- The sections (`h3` tags) displayed
- The structure of each section
- The number and format of listed elements

**The goal of this notebook is to explore and understand these variations in order to define a clean, scalable data schema.**

## Methodology

1. Extract raw HTML data for each deputy
2. Parse the `.stiri-detalii clearfix` container for each profile
3. From each parsed container, extract:
   - All section titles (`h3` tags)
   - The structure of their sub-elements (`p`, `a`, `table`, etc.)
4. Save the extracted structure into a JSON format for further analysis
5. Analyze and identify:
   - The unique list of section categories (`h3`)
   - The type of content found within each category
   - The frequency and variability of these sections across profiles


## Step 1 – Raw HTML Collection

*(This section includes the call to `get_deputies_list_selenium` and saving the raw container HTML in JSON format)*

In [4]:
import sys
sys.path.append("../src")

from scraper.get_deputies_details import get_deputies_details_selenium
import json
import os

In [11]:
os.makedirs("../data/raw", exist_ok=True)

deputies = get_deputies_details_selenium(max_steps=500)

with open("../data/raw/deputati_raw.json", "w", encoding="utf-8") as f:
    json.dump(deputies, f, ensure_ascii=False, indent=2)

## Step 2 – Preliminary Parsing

*(Here we parse each profile’s HTML, extract `h3` section headers, and capture the surrounding structure)*


In [6]:
with open("../data/raw/deputati_raw.json", "r", encoding="utf-8") as f:
    deputati_raw = json.load(f)

from bs4 import BeautifulSoup
from collections import defaultdict

def parse_deputat_html(html):
    soup = BeautifulSoup(html, "html.parser")
    sections = defaultdict(list)
    current_section = None

    for elem in soup.find_all(["h3", "p", "table", "ul", "ol", "div"]):
        if elem.name == "h3":
            current_section = elem.get_text(strip=True)
            sections[current_section] = []
        elif current_section:
            text = elem.get_text(" ", strip=True)
            if text:
                sections[current_section].append(text)
    return dict(sections)

parsed_profiles = [
    {
        "profil_link": dep["profil_link"],
        "sectiuni": parse_deputat_html(dep["data_details_container"])
    }
    for dep in deputati_raw
]


In [10]:
parsed_profiles.keys()

AttributeError: 'list' object has no attribute 'keys'

## Step 3 – Section Analysis

Questions we aim to answer:
- What `h3` sections appear across all profiles?
- How frequently does each section occur?
- Are there sections that appear only in specific profiles?


## Intermediate Observations

_(This is where we document findings — rare sections, inconsistencies, optional fields, etc.)_



## Next Steps

For each commonly occurring section:
- Explore the sub-structure (e.g., committees → link, role, date?)
- Define a specific parser for that section
- Decide whether the section should be modeled as:
  - A separate entity (e.g., `Committees`)
  - A flexible activity field (e.g., `Speeches`, `Legislative Proposals`)